In [0]:
# LOAD ALL 7 CSV FILES

orders      = spark.read.csv('/Volumes/workspace/default/dataset_olist/olist_orders_dataset.csv',
                             header=True, inferSchema=True)
items       = spark.read.csv('/Volumes/workspace/default/dataset_olist/olist_order_items_dataset.csv',
                             header=True, inferSchema=True)
customers   = spark.read.csv('/Volumes/workspace/default/dataset_olist/olist_customers_dataset.csv',
                             header=True, inferSchema=True)
payments    = spark.read.csv('/Volumes/workspace/default/dataset_olist/olist_order_payments_dataset.csv',
                             header=True, inferSchema=True)
reviews     = spark.read.csv('/Volumes/workspace/default/dataset_olist/olist_order_reviews_dataset.csv',
                             header=True, inferSchema=True)
products    = spark.read.csv('/Volumes/workspace/default/dataset_olist/olist_products_dataset.csv',
                             header=True, inferSchema=True)
translation = spark.read.csv('/Volumes/workspace/default/dataset_olist/product_category_name_translation.csv',
                             header=True, inferSchema=True)

print(f'Orders:      {orders.count():,} rows')
print(f'Items:       {items.count():,} rows')
print(f'Customers:   {customers.count():,} rows')
print(f'Payments:    {payments.count():,} rows')
print(f'Reviews:     {reviews.count():,} rows')
print(f'Products:    {products.count():,} rows')


In [0]:
# EXPLORE THE DATA

print('=== ORDERS SCHEMA ===')
orders.printSchema()

print('=== ORDERS FIRST 5 ROWS ===')
orders.show(5)

# Count null values in orders
from pyspark.sql.functions import col, sum as spark_sum
null_counts = orders.select([
  spark_sum(col(c).isNull().cast('int')).alias(c)
  for c in orders.columns
])
print('=== NULL VALUE COUNT PER COLUMN ===')
null_counts.show()


In [0]:
# DATA CLEANING

from pyspark.sql.functions import to_timestamp, datediff, col

# ORDERS: keep only delivered orders
orders_clean = orders.filter(col('order_status') == 'delivered')
print(f'Delivered orders: {orders_clean.count():,}')

# Drop rows with missing timestamps
orders_clean = orders_clean.dropna(subset=[
    'order_purchase_timestamp',
    'order_delivered_customer_date'
])
print(f'After dropping nulls: {orders_clean.count():,}')

# Remove duplicate order IDs
orders_clean = orders_clean.dropDuplicates(['order_id'])

# Calculate delivery days
orders_clean = orders_clean.withColumn(
    'delivery_days',
    datediff(
        to_timestamp(col('order_delivered_customer_date'), 'M/d/yyyy H:mm'),
        to_timestamp(col('order_purchase_timestamp'), 'M/d/yyyy H:mm')
    )
)

# Remove invalid delivery times
orders_clean = orders_clean.filter(col('delivery_days') > 0)
print(f'Final clean orders: {orders_clean.count():,}')

# REVIEWS: drop null review scores, one review per order
reviews_clean = reviews.dropna(subset=['review_score', 'order_id'])
reviews_clean = reviews_clean.dropDuplicates(['order_id'])
print(f'Reviews after cleaning: {reviews_clean.count():,}')

# PAYMENTS: drop null or zero payment values
payments_clean = payments.dropna(subset=['payment_value', 'order_id'])
payments_clean = payments_clean.filter(col('payment_value') > 0)
print(f'Payments after cleaning: {payments_clean.count():,}')

# ITEMS: drop null prices or negative freight values
items_clean = items.dropna(subset=['price', 'freight_value', 'order_id'])
items_clean = items_clean.filter((col('price') > 0) & (col('freight_value') >= 0))
print(f'Items after cleaning: {items_clean.count():,}')

# PRODUCTS: fill null category with 'unknown'
products_clean = products.fillna({'product_category_name': 'unknown'})
print(f'Products after cleaning: {products_clean.count():,}')

# CUSTOMERS: drop rows with null customer_state
customers_clean = customers.dropna(subset=['customer_state'])
print(f'Customers after cleaning: {customers_clean.count():,}')

In [0]:
# JOIN ALL TABLES INTO MASTER DATAFRAME

from pyspark.sql.functions import avg, sum as spark_sum, count, desc
from pyspark.sql.functions import round as spark_round

# Join orders + reviews (using reviews_clean)
df = orders_clean.join(
    reviews_clean.select('order_id', 'review_score'),
    on='order_id', how='inner'
)

# Aggregate payments per order (using payments_clean)
pay_agg = payments_clean.groupBy('order_id').agg(
    avg('payment_value').alias('avg_payment'),
    avg('payment_installments').alias('payment_installments')
)
df = df.join(pay_agg, on='order_id', how='left')

# Aggregate items per order (using items_clean)
items_agg = items_clean.groupBy('order_id').agg(
    spark_sum('price').alias('total_price'),
    spark_sum('freight_value').alias('freight_value'),
    count('order_item_id').alias('item_count')
)
df = df.join(items_agg, on='order_id', how='left')

# Add customer state (using customers_clean)
df = df.join(
    customers_clean.select('customer_id', 'customer_state'),
    on='customer_id', how='left'
)

# Add product category in English (using items_clean and products_clean)
items_cat = items_clean.join(
    products_clean.select('product_id', 'product_category_name'),
    on='product_id', how='left'
).join(translation, on='product_category_name', how='left'
).select('order_id', 'product_category_name_english'
).dropDuplicates(['order_id'])

df = df.join(items_cat, on='order_id', how='left')

# Drop any remaining nulls in key feature columns used for ML later
df = df.dropna(subset=['total_price', 'freight_value',
                        'payment_installments', 'item_count',
                        'delivery_days', 'review_score'])

print(f'Master DataFrame: {df.count():,} rows, {len(df.columns)} columns')
df.show(3)

In [0]:
# ANALYTICS: REVENUE BY PRODUCT CATEGORY

revenue_by_cat = df.groupBy('product_category_name_english').agg(
  spark_round(spark_sum('total_price'), 2).alias('total_revenue'),
  count('order_id').alias('total_orders'),
  spark_round(avg('total_price'), 2).alias('avg_order_value')
).orderBy(desc('total_revenue'))

print('=== TOP 10 PRODUCT CATEGORIES BY REVENUE ===')
revenue_by_cat.show(10)


In [0]:
# ANALYTICS: MONTHLY ORDER TRENDS

from pyspark.sql.functions import date_format, to_timestamp

monthly = df.withColumn(
    'month',
    date_format(
        to_timestamp(col('order_purchase_timestamp'), 'M/d/yyyy H:mm'),  # ✅ format added
        'yyyy-MM'
    )
).groupBy('month').agg(
    count('order_id').alias('order_count'),
    spark_round(spark_sum('total_price'), 2).alias('monthly_revenue')
).orderBy('month')

print('=== MONTHLY ORDER VOLUME AND REVENUE ===')
monthly.show(24)

In [0]:
# ANALYTICS: AVERAGE DELIVERY TIME BY STATE

delivery_by_state = df.groupBy('customer_state').agg(
  spark_round(avg('delivery_days'), 1).alias('avg_delivery_days'),
  count('order_id').alias('total_orders')
).orderBy(desc('avg_delivery_days'))

print('=== DELIVERY PERFORMANCE BY STATE (Slowest first) ===')
delivery_by_state.show(15)

In [0]:
# ANALYTICS: REVIEW SCORE AND PAYMENT DISTRIBUTION

review_dist = df.groupBy('review_score').agg(
  count('order_id').alias('count')
).orderBy('review_score')
print('=== REVIEW SCORE DISTRIBUTION ===')
review_dist.show()

pay_dist = payments.groupBy('payment_type').agg(
  count('order_id').alias('count')
).orderBy(desc('count'))
print('=== PAYMENT METHOD DISTRIBUTION ===')
pay_dist.show()
